#### Homework 07:  Parallel Programming

## Due Date: Apr 17, 2026, 11:00pm

#### Firstname Lastname: Dayne Lee

#### E-mail: dl5635@nyu.edu

#### Enter your solutions and submit this notebook

---

**Problem 1 (40p)**

In this problem the goal is to calculate the mean and standard deviation of a large list of numbers by using Reduction as one of Collective Operations, see Lecture 11. 


Consider $N = 256000$ random variables uniform on $[0, 1]$, call these $x_0, x_1, \dots, x_{N - 1}$.  


Write an MPI program with $N=16$ processes that outputs the average and standard deviation of $x_0, x_1, \dots, x_{N - 1}$.


To simplify the problem, let one process, say `Process 0`, independently draws $N$ samples uniformly on $[0, 1]$.

How do you explain the results?


**Instructions:** 
Your program should use MPI4PY and collective operations. 
Save your program as 2026_spring_sol07_pr01.py and run it from the terminal as:

```
mpirun -n 16 python 2026_spring_sol07_pr01.py
```


In [1]:
%%writefile 2026_spring_sol07_pr01.py

from mpi4py import MPI
import numpy as np

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

N = 256000

# --- Step 1: root generates data ---
if rank == 0:
    data = np.random.uniform(0.0, 1.0, N)
else:
    data = None

# --- Step 2: scatter data ---
local_n = N // size
local_data = np.zeros(local_n)

comm.Scatter(data, local_data, root=0)

# --- Step 3: compute local sums ---
local_sum = np.sum(local_data)
local_sq_sum = np.sum(local_data**2)

# --- Step 4: reduce to global sums ---
total_sum = comm.reduce(local_sum, op=MPI.SUM, root=0)
total_sq_sum = comm.reduce(local_sq_sum, op=MPI.SUM, root=0)

# --- Step 5: compute mean and std on root ---
if rank == 0:
    mean = total_sum / N
    variance = (total_sq_sum / N) - mean**2
    std = np.sqrt(variance)

    print(f"Mean = {mean}")
    print(f"Std  = {std}")

Overwriting 2026_spring_sol07_pr01.py


In [2]:
# !mpiexec -n 16 python 2026_spring_sol07_pr01.py
# !mpirun -n 16 python 2026_spring_sol07_pr01.py

!sysctl -n hw.ncpu
# since the number of processes is greater than the number of cores, we need to oversubscribe
!mpirun -n 16 --oversubscribe python 2026_spring_sol07_pr01.py

10
Mean = 0.501120993980253
Std  = 0.2885028407707779


**Q. How do you explain the results?**

A. Due to the law of large numbers, the computed mean approaches 0.5 and the standard deviation approaches $\sqrt{\frac{1}{12}} \approx 0.2887$, the theoretical values for a uniform distribution on [0,1]. Small differences are due to randomness and finite sample size.


---
**Problem 2 (60p)**

In this problem the goal is to demonstrate how one can use a Domain Decomposition and  Collective Operations. 

Consider the exponential distribution $X \sim \textrm{Exp}(1)$ with the unit mean. Find numerical approximations of moments of the exponential random variable. 

That is, for a random variable $X$ with the distribution $f(x) = e^{-x}$ for $x \geq 0$, compute the first $15$ moments, where the $k$-th moment is defined as:
$$I_k = \int_{0}^{\infty} x^k f(x) dx.$$


Your program should use MPI parallel collective instructions, where the integration is performed in parallel over $N=16$ processes, over a finite range $[0, M)$, where $M=1000$, with $N = 16$ partitions and $1000$ increments per partition, see Lecture 10 and 11.

Provide evaluations of $J_1, J_2, \dots, J_{15}$, where $$J_k = \int_{0}^{M} x^k f(x) dx.$$


**Instructions:** 

Save your program as 2026_spring_sol07_pr02.py; and run it from the terminal as:

```
mpirun -n 16 python 2026_spring_sol07_pr02.py
```


**Bonus Question (10 points):** 

What is the value of $I_k$, as a function of $k$? How can it be derived?

In [3]:
%%writefile 2026_spring_sol07_pr02.py

from mpi4py import MPI
import numpy as np
from math import exp, factorial

comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()

# Problem settings
M = 1000.0          # finite integration range [0, M]
n_local = 1000      # increments per partition
k_max = 15          # compute J1, J2, ..., J15

# Domain decomposition
# Total domain [0, M] is split into `size` partitions
local_width = M / size
h = local_width / n_local

a_local = rank * local_width
b_local = a_local + local_width

# Local contribution for J1 ... J15
local_J = np.zeros(k_max, dtype=np.float64)

# Midpoint rule on this rank's subinterval
for j in range(n_local):
    x = a_local + (j + 0.5) * h
    fx = exp(-x)

    # x shape: scalar
    # fx shape: scalar
    # local_J shape: (15,)
    x_pow = x
    for k in range(1, k_max + 1):
        local_J[k - 1] += x_pow * fx * h
        x_pow *= x

# Reduce all local partial integrals to root
global_J = np.zeros(k_max, dtype=np.float64)
comm.Reduce(local_J, global_J, op=MPI.SUM, root=0)

if rank == 0:
    print(f"Computed moments J_k (from 0 to {M:.0f})")
    print(f"Processes = {size}, partitions = {size}, increments/partition = {n_local}")
    print()

    for k in range(1, k_max + 1):
        exact = factorial(k)   # I_k == k!
        approx = global_J[k - 1]
        rel_err = abs(approx - exact) / exact
        print(f"J_{k:2d} = {approx: .10e}   exact = {exact: .10e}   rel.err = {rel_err: .3e}")

Overwriting 2026_spring_sol07_pr02.py


In [4]:
!mpirun -n 16 --oversubscribe python 2026_spring_sol07_pr02.py

Computed moments J_k (from 0 to 1000)
Processes = 16, partitions = 16, increments/partition = 1000

J_ 1 =  1.0001627048e+00   exact =  1.0000000000e+00   rel.err =  1.627e-04
J_ 2 =  2.0000001112e+00   exact =  2.0000000000e+00   rel.err =  5.561e-08
J_ 3 =  5.9999998889e+00   exact =  6.0000000000e+00   rel.err =  1.852e-08
J_ 4 =  2.4000000000e+01   exact =  2.4000000000e+01   rel.err =  9.541e-12
J_ 5 =  1.2000000000e+02   exact =  1.2000000000e+02   rel.err =  1.904e-12
J_ 6 =  7.2000000000e+02   exact =  7.2000000000e+02   rel.err =  1.579e-16
J_ 7 =  5.0400000000e+03   exact =  5.0400000000e+03   rel.err =  1.805e-16
J_ 8 =  4.0320000000e+04   exact =  4.0320000000e+04   rel.err =  9.023e-16
J_ 9 =  3.6288000000e+05   exact =  3.6288000000e+05   rel.err =  4.812e-16
J_10 =  3.6288000000e+06   exact =  3.6288000000e+06   rel.err =  6.416e-16
J_11 =  3.9916800000e+07   exact =  3.9916800000e+07   rel.err =  7.466e-16
J_12 =  4.7900160000e+08   exact =  4.7900160000e+08   rel.err =

**Q. What is the value of $I_k$, as a function of $k$? How can it be derived?**


The value is $I_k=\int_0^\infty x^k e^{-x}dx = k!$. It is derived by integration by parts, which gives the recurrence $I_k = k I_{k-1}$, together with the base case $I_0=1$.

Below is the derivation:


$$
I_k=\int_0^\infty x^k e^{-x}\,dx
$$

Using integration by parts, let

$$
u=x^k,\qquad dv=e^{-x}\,dx
$$

Then

$$
du=kx^{k-1}\,dx,\qquad v=-e^{-x}
$$

So,

$$
I_k=\left[-x^k e^{-x}\right]_0^\infty + k\int_0^\infty x^{k-1}e^{-x}\,dx
$$

Since $\left[-x^k e^{-x}\right]_0^\infty=0$, we get the recurrence

$$
I_k = k I_{k-1}
$$

with base case

$$
I_0=\int_0^\infty e^{-x}\,dx=1
$$

Therefore,

$$
I_k = k(k-1)(k-2)\cdots 1 = k!
$$
